## Installation - Required Packages
Install Optuna for hyperparameter optimization.

In [1]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 10.6 MB/s eta 0:00:00


## Installation - Kaggle Hub
Install kagglehub to download datasets from Kaggle.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ! pip install kagglehub[pandas-datasets]

## Installation - TorchInfo
Install torchinfo for model architecture visualization.

In [3]:
! pip install torchinfo

## Import Libraries and Configure Settings
Import all necessary libraries for data manipulation, visualization, machine learning, and deep learning. Set random seeds for reproducibility.

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot  as plt
import seaborn as sns
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
import optuna
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from imblearn.over_sampling import SMOTE

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)


## Device Configuration
Check and set the computation device to GPU if available, otherwise use CPU.

In [5]:
# Check for GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


## Dataset Loading from Kaggle
Download and load the AI4I 2020 Predictive Maintenance Dataset from Kaggle.

In [ ]:
# import kagglehub
# from kagglehub import KaggleDatasetAdapter

# # Set the path to the file you'd like to load
# # For this dataset, the main file is 'ai4i2020.csv'
# file_path = "ai4i2020.csv"

# # Load the latest version using the recommended dataset_load method
# df = kagglehub.dataset_load(
#   KaggleDatasetAdapter.PANDAS,
#   "stephanmatzka/predictive-maintenance-dataset-ai4i-2020",
#   file_path
# )

## Save Dataset to Cloud Storage
Save the loaded dataset to Google Drive for persistent storage.

In [ ]:
# # Save to your Drive
# df.to_csv('/content/drive/MyDrive/Colab Notebooks/Predictive Maintenance Dataset (AI4I 2020)/ai4i2020_saved.csv', index=False)
# print("✅ Saved to Google Drive!")

## Mount Google Drive
Connect to Google Drive to access saved datasets.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

Mounted at /content/drive


## Load and Inspect Dataset
Read the CSV file from Google Drive and display the first few rows to understand the data structure.

In [76]:
df=pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Predictive Maintenance Dataset (AI4I 2020)/ai4i2020_saved.csv')
df.head()

,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


## Dataset Dimensions
Check the number of rows and columns in the dataset.

In [77]:
df.shape

(10000, 14)

## Feature Selection - Remove Non-Predictive Columns
Drop columns like UDI (unique identifier) and Product ID that don't contribute to prediction.

In [78]:
# Drop non-predictive columns
df = df.drop(columns=['UDI', 'Product ID'])

## Check for Missing Values and Data Types
Inspect the dataset for any missing values and review data types of all features.

In [79]:
pd.DataFrame({"Missing value":df.isna().sum(),"Datatype":df.dtypes})

,Missing value,Datatype
Type,0,object
Air temperature [K],0,float64
Process temperature [K],0,float64
Rotational speed [rpm],0,int64
Torque [Nm],0,float64
Tool wear [min],0,int64
Machine failure,0,int64
TWF,0,int64
HDF,0,int64
PWF,0,int64


## Prepare Features (X)
Separate features (X) from the target variable by dropping 'Machine failure' column.

In [80]:
X = df.drop('Machine failure', axis=1)
X.head()

,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],TWF,HDF,PWF,OSF,RNF
0,M,298.1,308.6,1551,42.8,0,0,0,0,0,0
1,L,298.2,308.7,1408,46.3,3,0,0,0,0,0
2,L,298.1,308.5,1498,49.4,5,0,0,0,0,0
3,L,298.2,308.6,1433,39.5,7,0,0,0,0,0
4,L,298.2,308.7,1408,40.0,9,0,0,0,0,0


## Prepare Target Variable (y)
Extract the target variable 'Machine failure' for supervised learning.

In [81]:
y=df["Machine failure"]
y.head()

,Machine failure
0,0
1,0
2,0
3,0
4,0


## Analyze Class Distribution
Check the distribution of the target variable to identify class imbalance.

In [82]:
df["Machine failure"].value_counts()

,count
Machine failure,
0,9661
1,339


### This is highly imbalanced data for this we should balance the data using SMOTE

## Train-Test Split with Stratification
Split the data into training (80%) and testing (20%) sets while preserving the class distribution ratio.

In [83]:
# Stratified split to preserve failure ratio
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

## Encode Categorical Features
Apply Label Encoding to the 'Type' categorical feature to convert it into numerical format.

In [84]:
# Encode categorical feature
le = LabelEncoder()
X_train['Type'] = le.fit_transform(X_train['Type'])
X_test['Type'] = le.transform(X_test['Type'])

## Handle Class Imbalance with SMOTE
Apply Synthetic Minority Over-sampling Technique (SMOTE) to balance the training dataset.

In [85]:
# Apply SMOTE to training data only
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)


## Feature Scaling
Standardize features to have zero mean and unit variance for better neural network training.

In [86]:
# Scale features
scaler = StandardScaler()
X_train_resampled = scaler.fit_transform(X_train_resampled)
X_test = scaler.transform(X_test)

## Verify SMOTE Balancing
Check the class distribution after applying SMOTE to ensure balanced classes.

In [87]:
y_train_resampled.value_counts()

,count
Machine failure,
0,7729
1,7729


## Custom Dataset Class
Define a PyTorch Dataset class to handle feature-label pairs and enable efficient data loading.

In [88]:
# Custom Dataset and data loader
class CustomDataset(Dataset):
    def __init__(self,features,labels):
        self.features = torch.tensor(features,dtype=torch.float32)
        self.labels = torch.tensor(labels.values, dtype=torch.long)
    def __len__(self):
        return len(self.features)
    def __getitem__(self, index):
        return self.features[index],self.labels[index]

## Initialize Training Dataset
Create a CustomDataset instance for the training data with resampled features.

In [89]:
train_dataset = CustomDataset(X_train_resampled, y_train_resampled)

## Initialize Test Dataset
Create a CustomDataset instance for the test data.

In [90]:
test_dataset = CustomDataset(X_test, y_test)

## Create DataLoaders
Set up DataLoaders with specified batch size for efficient training and evaluation.

In [91]:
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,pin_memory=True)
test_loader  = DataLoader(test_dataset, batch_size=batch_size, shuffle=False,pin_memory=True)

## Verify DataLoader Size
Check the number of batches in the training DataLoader.

In [92]:
len(train_loader)

484

## Define Neural Network Architecture
Create a simple feedforward neural network (ANN) with two hidden layers, ReLU activations, and dropout for regularization.

In [93]:
class PredictiveMaintenanceANN(nn.Module):
    def __init__(self,num_features):
        super().__init__()
        self.model=nn.Sequential(
            nn.Linear(num_features, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 1)
        )
    def forward(self,x):
        return self.model(x)

## Configure Training Hyperparameters
Set the number of training epochs and initial learning rate.

In [94]:
# set learning rate and epochs
epochs = 100
learning_rate = 0.001

## Initialize Model
Instantiate the neural network model and move it to the appropriate device (GPU/CPU).

In [95]:
# instatiate the model
model=PredictiveMaintenanceANN(X_train_resampled.shape[1])
model=model.to(device)
model

PredictiveMaintenanceANN(
  (model): Sequential(
    (0): Linear(in_features=11, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=128, out_features=64, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.3, inplace=False)
    (6): Linear(in_features=64, out_features=1, bias=True)
  )
)

## Model Summary
Display the model architecture, parameter count, and computational complexity using torchinfo.

In [96]:
from torchinfo import summary

summary(model, input_size=(1, 11))

Layer (type:depth-idx)                   Output Shape              Param #
PredictiveMaintenanceANN                 [1, 1]                    --
├─Sequential: 1-1                        [1, 1]                    --
│    └─Linear: 2-1                       [1, 128]                  1,536
│    └─ReLU: 2-2                         [1, 128]                  --
│    └─Dropout: 2-3                      [1, 128]                  --
│    └─Linear: 2-4                       [1, 64]                   8,256
│    └─ReLU: 2-5                         [1, 64]                   --
│    └─Dropout: 2-6                      [1, 64]                   --
│    └─Linear: 2-7                       [1, 1]                    65
Total params: 9,857
Trainable params: 9,857
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 0.01
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 0.04
Estimated Total Size (MB): 0.04

## Define Loss Function and Optimizer
Set up Binary Cross-Entropy loss with logits and the AdamW optimizer for training.

In [97]:
# Loss Function
criterion = nn.BCEWithLogitsLoss()

# optimizer
optimizer = optim.AdamW(model.parameters(), lr=learning_rate)


## Model Training Loop
Train the neural network for the specified number of epochs, computing forward pass, loss, backpropagation, and weight updates.

In [98]:
# training loop

for epoch in range(epochs):

    total_epoch_loss = 0

    for batch_features,batch_labels in train_loader:

        batch_features,batch_labels=batch_features.to(device),batch_labels.to(device)

        # forward pass
        outputs = model(batch_features).view(-1)

        # calculate loss
        loss = criterion(outputs, batch_labels.float())

        # back pass
        optimizer.zero_grad()

        loss.backward()

        # update grads
        optimizer.step()

        total_epoch_loss = total_epoch_loss + loss.item()
    avg_loss = total_epoch_loss/len(train_loader)
    print(f'Epoch: {epoch + 1} , Loss: {avg_loss}')

Epoch: 1 , Loss: 0.21076214410283328
Epoch: 2 , Loss: 0.1495095679232435
Epoch: 3 , Loss: 0.13694319230589
Epoch: 4 , Loss: 0.13474146190892197
Epoch: 5 , Loss: 0.1306419897947676
Epoch: 6 , Loss: 0.1237892693897273
Epoch: 7 , Loss: 0.12331977850095607
Epoch: 8 , Loss: 0.11821129952734413
Epoch: 9 , Loss: 0.11613148894720934
Epoch: 10 , Loss: 0.11507231703115785
Epoch: 11 , Loss: 0.11189818457692616
Epoch: 12 , Loss: 0.11083589457212305
Epoch: 13 , Loss: 0.10915981411495176
Epoch: 14 , Loss: 0.10767788700937402
Epoch: 15 , Loss: 0.10627345851028877
Epoch: 16 , Loss: 0.10463288032848095
Epoch: 17 , Loss: 0.10346450795413437
Epoch: 18 , Loss: 0.10491035632165688
Epoch: 19 , Loss: 0.1007363564432945
Epoch: 20 , Loss: 0.10040447944852192
Epoch: 21 , Loss: 0.09680995987629697
Epoch: 22 , Loss: 0.09751652705458222
Epoch: 23 , Loss: 0.09487435945178851
Epoch: 24 , Loss: 0.09402164653586301
Epoch: 25 , Loss: 0.09441463027653585
Epoch: 26 , Loss: 0.09206047778648963
Epoch: 27 , Loss: 0.09321531

## Set Model to Evaluation Mode
Switch the model from training mode to evaluation mode to disable dropout and enable correct behavior for inference.

In [99]:
# set model to eval mode
model.eval()

PredictiveMaintenanceANN(
  (model): Sequential(
    (0): Linear(in_features=11, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=128, out_features=64, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.3, inplace=False)
    (6): Linear(in_features=64, out_features=1, bias=True)
  )
)

## Model Evaluation on Test Set
Evaluate the trained model on the test dataset and compute performance metrics including accuracy, precision, recall, F1-score, and confusion matrix.

In [100]:
# evaluation code
preds, labels = [], []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch,y_batch=X_batch.to(device),y_batch.to(device)

        logits = model(X_batch).squeeze()                 # Forward pass
        probs = torch.sigmoid(logits).cpu().numpy()       # Convert logits → probabilities
        preds.extend(probs)                               # Store for later
        labels.extend(y_batch.cpu().numpy())

preds = np.array(preds)
labels = np.array(labels)
y_pred = (np.array(preds) >= 0.5).astype(int)             # Threshold to 0/1 predictions

print("\n📊 Test Metrics:")
print(f"Accuracy:  {accuracy_score(labels, y_pred)}")
print(f"Precision: {precision_score(labels, y_pred)}")
print(f"Recall:    {recall_score(labels, y_pred)}")
print(f"F1-Score:  {f1_score(labels, y_pred)}")
print("\n📋 Confusion Matrix:")
cm = confusion_matrix(labels, y_pred)
cm



📊 Test Metrics:
Accuracy:  0.9645
Precision: 0.48905109489051096
Recall:    0.9852941176470589
F1-Score:  0.6536585365853659

📋 Confusion Matrix:


array([[1862,   70],
       [   1,   67]])

## Hyperparameter Tuning with Optuna

## Define Flexible Neural Network Architecture
Create a configurable neural network class that supports variable numbers of hidden layers, neurons, dropout rates, and batch normalization for hyperparameter tuning.

In [26]:
class PredictiveHyperANN(nn.Module):

    def __init__(self,input_dim,output_dim,num_hidden_layes,neurons_per_layers,dropout_rate):
        super().__init__()

        layers=[]

        for i in range(num_hidden_layes):

            layers.append(nn.Linear(input_dim,neurons_per_layers))
            layers.append(nn.BatchNorm1d(neurons_per_layers))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout_rate))
            input_dim=neurons_per_layers
        layers.append(nn.Linear(neurons_per_layers,output_dim))

        self.model=nn.Sequential(*layers)

    def forward(self,x):
        return self.model(x)

## Optuna Objective Function
Define the objective function for Optuna hyperparameter optimization. This function tests different hyperparameter combinations including number of layers, neurons, learning rate, dropout, batch size, optimizer type, and weight decay with early stopping.

In [27]:
# Objective Function

def objective(trial):

    num_hidden_layers=trial.suggest_int("num_hidden_layers",1,8)
    neurons_per_layer = trial.suggest_int("neurons_per_layer", 8, 128, step=8)
    epochs = trial.suggest_int("epochs", 10, 100, step=10)
    learning_rate = trial.suggest_float("learning_rate", 1e-5, 1e-1, log=True)
    dropout_rate = trial.suggest_float("dropout_rate", 0.1, 0.5, step=0.1)
    batch_size = trial.suggest_categorical("batch_size", [16, 32, 64, 128])
    optimizer_name = trial.suggest_categorical("optimizer", ['Adam', 'SGD', 'RMSprop'])
    weight_decay = trial.suggest_float("weight_decay", 1e-5, 1e-3, log=True)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, pin_memory=True)
    test_loader  = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, pin_memory=True)

    # Model init
    input_dim = X_train_resampled.shape[1]
    output_dim = 1

    # Create model with trial-specific hyperparameters
    model = PredictiveHyperANN(input_dim, output_dim, num_hidden_layers, neurons_per_layer, dropout_rate).to(device)

    # Optimizer Selection
    criterion = nn.BCEWithLogitsLoss()

    if optimizer_name == "Adam":
        optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    elif optimizer_name == "SGD":
        optimizer = optim.SGD(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    elif optimizer_name == "RMSprop":
        optimizer = optim.RMSprop(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

    # Early stopping parameters
    patience = 7  # how many epochas to wait without improvement
    best_val_acc = 0 # tracks the highest accuracy so far
    patience_counter = 0 # Counts epochs since last improvement
    best_model_state = None # Stores the best model weights

    for epoch in range(epochs):
        # Training
        model.train()
        for batch_features, batch_labels in train_loader:
            batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)

            # Forward pass
            outputs = model(batch_features).view(-1)

            # Calculate loss
            loss = criterion(outputs, batch_labels.float())

            # Backward pass
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        # Evaluation after each epoch
        model.eval()
        pred = []
        true_labels = []

        with torch.no_grad():
            for X_batch, y_batch in test_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                logits = model(X_batch).view(-1)
                prob = torch.sigmoid(logits).cpu().numpy()
                pred.extend(prob)
                true_labels.extend(y_batch.cpu().numpy())

        y_pred = (np.array(pred) >= 0.5).astype(int)
        accuracy = accuracy_score(true_labels, y_pred)

        # Report intermediate result to Optuna
        trial.report(accuracy, epoch) # Sends the current epoch's accuracy to Optuna.

        # Early stopping check
        if accuracy > best_val_acc:
            best_val_acc = accuracy
            patience_counter = 0
            best_model_state = model.state_dict().copy()
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"Early stopping at epoch {epoch+1}, best accuracy: {best_val_acc}")
                break

        # Handle pruning by Optuna
        if trial.should_prune():
            raise optuna.TrialPruned()

    # Load best model state if available
    if best_model_state is not None:
        model.load_state_dict(best_model_state)

    return best_val_acc


In [61]:
study=optuna.create_study(study_name="hyperparameter_optimization",direction="maximize",sampler=optuna.samplers.TPESampler())

study.optimize(objective,n_trials=20)

[I 2026-04-09 07:47:55,260] A new study created in memory with name: hyperparameter_optimization
[I 2026-04-09 07:48:07,731] Trial 0 finished with value: 0.89 and parameters: {'num_hidden_layers': 1, 'neurons_per_layer': 16, 'epochs': 30, 'learning_rate': 2.4749781540323197e-05, 'dropout_rate': 0.4, 'batch_size': 128, 'optimizer': 'RMSprop', 'weight_decay': 0.0006828113352301822}. Best is trial 0 with value: 0.89.
[I 2026-04-09 07:48:14,356] Trial 1 finished with value: 0.9935 and parameters: {'num_hidden_layers': 8, 'neurons_per_layer': 96, 'epochs': 20, 'learning_rate': 4.42629080410185e-05, 'dropout_rate': 0.2, 'batch_size': 128, 'optimizer': 'RMSprop', 'weight_decay': 1.7104046243971686e-05}. Best is trial 1 with value: 0.9935.


Early stopping at epoch 9, best accuracy: 0.9935


[I 2026-04-09 07:49:23,011] Trial 2 finished with value: 0.98 and parameters: {'num_hidden_layers': 8, 'neurons_per_layer': 64, 'epochs': 100, 'learning_rate': 0.00018836858900545553, 'dropout_rate': 0.2, 'batch_size': 32, 'optimizer': 'SGD', 'weight_decay': 4.1562551244620994e-05}. Best is trial 1 with value: 0.9935.


Early stopping at epoch 23, best accuracy: 0.98


[I 2026-04-09 07:49:36,602] Trial 3 finished with value: 0.9895 and parameters: {'num_hidden_layers': 7, 'neurons_per_layer': 24, 'epochs': 10, 'learning_rate': 0.046376086779106755, 'dropout_rate': 0.4, 'batch_size': 64, 'optimizer': 'Adam', 'weight_decay': 1.7424605540771827e-05}. Best is trial 1 with value: 0.9935.
[I 2026-04-09 07:50:00,548] Trial 4 finished with value: 0.9975 and parameters: {'num_hidden_layers': 7, 'neurons_per_layer': 104, 'epochs': 40, 'learning_rate': 0.03297714094287559, 'dropout_rate': 0.1, 'batch_size': 32, 'optimizer': 'Adam', 'weight_decay': 0.0009132360144930952}. Best is trial 4 with value: 0.9975.


...

Early stopping at epoch 8, best accuracy: 0.9925


[I 2026-04-09 07:51:08,820] Trial 13 pruned. 
[I 2026-04-09 07:51:11,223] Trial 14 pruned. 
[I 2026-04-09 07:51:13,176] Trial 15 pruned. 
[I 2026-04-09 07:51:52,865] Trial 16 finished with value: 0.991 and parameters: {'num_hidden_layers': 7, 'neurons_per_layer': 120, 'epochs': 30, 'learning_rate': 0.0012964821725016018, 'dropout_rate': 0.30000000000000004, 'batch_size': 16, 'optimizer': 'RMSprop', 'weight_decay': 0.00015061837009017987}. Best is trial 10 with value: 0.9985.


Early stopping at epoch 8, best accuracy: 0.991


[I 2026-04-09 07:51:54,391] Trial 17 pruned. 
[I 2026-04-09 07:51:56,990] Trial 18 pruned. 
[I 2026-04-09 07:52:56,260] Trial 19 finished with value: 0.9985 and parameters: {'num_hidden_layers': 6, 'neurons_per_layer': 88, 'epochs': 40, 'learning_rate': 0.014011852556739191, 'dropout_rate': 0.30000000000000004, 'batch_size': 16, 'optimizer': 'RMSprop', 'weight_decay': 4.448731846220454e-05}. Best is trial 10 with value: 0.9985.


Early stopping at epoch 13, best accuracy: 0.9985


## Import Optuna Visualization Tools
Import plotting functions from Optuna for visualizing optimization results.

In [62]:
# For visualizations
from optuna.visualization import plot_optimization_history, plot_parallel_coordinate, plot_slice, plot_contour, plot_param_importances

## Optimization History Plot
Visualize how the optimization progressed over trials, showing the best objective value over time.

In [101]:
# 1. Optimization History
plot_optimization_history(study).show()

## Parallel Coordinates Plot
Visualize the relationship between hyperparameters and objective values in parallel coordinates.

In [102]:
# 2. Parallel Coordinates Plot
plot_parallel_coordinate(study).show()

## Slice Plot
Show the relationship between individual hyperparameters and the objective value.

In [104]:
# 3. Slice Plot
plot_slice(study).show()

## Contour Plot
Visualize the interaction between pairs of hyperparameters and their effect on the objective value.

In [105]:
# 4. Contour Plot
plot_contour(study).show()

## Hyperparameter Importance Plot
Show which hyperparameters have the most impact on the model performance.

In [106]:
# 5. Hyperparameter Importance
plot_param_importances(study).show()

## Save Best Model
Extract the best hyperparameters from Optuna, retrain the model, and save it along with metadata including scaler parameters, feature columns, and encoder classes.

In [64]:
# SAVE THE BEST MODEL

import json

# Get best trial parameters
best_params = study.best_params
best_accuracy = study.best_value

print(f"Best Trial: {study.best_trial.number}")
print(f"Best Accuracy: {best_accuracy:.4f}")
print(f"Best Parameters: {best_params}")

# Recreate the best model with best hyperparameters
best_model = PredictiveHyperANN(
    input_dim=X_train_resampled.shape[1],
    output_dim=1,
    num_hidden_layes=int(best_params['num_hidden_layers']),
    neurons_per_layers=int(best_params['neurons_per_layer']),
    dropout_rate=best_params['dropout_rate']
).to(device)

# Train the best model with best hyperparameters
best_train_loader = DataLoader(train_dataset, batch_size=int(best_params['batch_size']), shuffle=True, pin_memory=True)

if best_params['optimizer'] == "Adam":
    best_optimizer = optim.Adam(best_model.parameters(), lr=best_params['learning_rate'], weight_decay=best_params['weight_decay'])
elif best_params['optimizer'] == "SGD":
    best_optimizer = optim.SGD(best_model.parameters(), lr=best_params['learning_rate'], weight_decay=best_params['weight_decay'])
elif best_params['optimizer'] == "RMSprop":
    best_optimizer = optim.RMSprop(best_model.parameters(), lr=best_params['learning_rate'], weight_decay=best_params['weight_decay'])

criterion = nn.BCEWithLogitsLoss()

# Train for the number of epochs from best trial
best_epochs = int(best_params['epochs'])
for epoch in range(best_epochs):
    best_model.train()
    for batch_features, batch_labels in best_train_loader:
        batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)
        outputs = best_model(batch_features).view(-1)
        loss = criterion(outputs, batch_labels.float())
        best_optimizer.zero_grad()
        loss.backward()
        best_optimizer.step()

print(f"\n✅ Best model trained for {best_epochs} epochs")

Best Trial: 10
Best Accuracy: 0.9985
Best Parameters: {'num_hidden_layers': 6, 'neurons_per_layer': 128, 'epochs': 50, 'learning_rate': 0.09864450681023856, 'dropout_rate': 0.1, 'batch_size': 32, 'optimizer': 'Adam', 'weight_decay': 0.00021180280406223997}

✅ Best model trained for 50 epochs


In [65]:
from torchinfo import summary
summary(best_model,input=(1,11))

Layer (type:depth-idx)                   Param #
PredictiveHyperANN                       --
├─Sequential: 1-1                        --
│    └─Linear: 2-1                       1,536
│    └─BatchNorm1d: 2-2                  256
│    └─ReLU: 2-3                         --
│    └─Dropout: 2-4                      --
│    └─Linear: 2-5                       16,512
│    └─BatchNorm1d: 2-6                  256
│    └─ReLU: 2-7                         --
│    └─Dropout: 2-8                      --
│    └─Linear: 2-9                       16,512
│    └─BatchNorm1d: 2-10                 256
│    └─ReLU: 2-11                        --
│    └─Dropout: 2-12                     --
│    └─Linear: 2-13                      16,512
│    └─BatchNorm1d: 2-14                 256
│    └─ReLU: 2-15                        --
│    └─Dropout: 2-16                     --
│    └─Linear: 2-17                      16,512
│    └─BatchNorm1d: 2-18                 256
│    └─ReLU: 2-19                        --
│  

In [66]:
# Save model and metadata
model_checkpoint = {
    'model_state_dict': best_model.state_dict(), # Trained weights & biases
    'model_params': best_params, # Optuna's winning hyperparameters
    'input_dim': X_train_resampled.shape[1],
    'accuracy': best_accuracy,
    'scaler_mean': scaler.mean_.tolist(),
    'scaler_scale': scaler.scale_.tolist(),
    'feature_columns': X.columns.tolist(), # Column names in correct order
    'type_encoder_classes': le.classes_.tolist() # LabelEncoder categories (['L', 'M', 'H'])
}

torch.save(model_checkpoint, '/content/drive/MyDrive/Colab Notebooks/Predictive Maintenance Dataset (AI4I 2020)/best_predictive_model.pth')
print("✅ Model saved to 'best_predictive_model.pth'")

✅ Model saved to 'best_predictive_model.pth'


In [53]:
best_test_loader = DataLoader(test_dataset, batch_size=int(best_params['batch_size']), shuffle=False, pin_memory=True)


In [67]:
# evaluation code
best_test_loader = DataLoader(test_dataset, batch_size=int(best_params['batch_size']), shuffle=False, pin_memory=True)

preds, labels = [], []

with torch.no_grad():
    best_model.eval()
    for X_batch, y_batch in best_test_loader:
        X_batch,y_batch=X_batch.to(device),y_batch.to(device)

        logits = best_model(X_batch).squeeze()                 # Forward pass
        probs = torch.sigmoid(logits).cpu().numpy()       # Convert logits → probabilities
        preds.extend(probs)                               # Store for later
        labels.extend(y_batch.cpu().numpy())

preds = np.array(preds)
labels = np.array(labels)
y_pred = (np.array(preds) >= 0.5).astype(int)             # Threshold to 0/1 predictions

print("\n📊 Test Metrics:")
print(f"Accuracy:  {accuracy_score(labels, y_pred)}")
print(f"Precision: {precision_score(labels, y_pred)}")
print(f"Recall:    {recall_score(labels, y_pred)}")
print(f"F1-Score:  {f1_score(labels, y_pred)}")
print("\n📋 Confusion Matrix:")
cm = confusion_matrix(labels, y_pred)
cm



📊 Test Metrics:
Accuracy:  0.9815
Precision: 0.6534653465346535
Recall:    0.9705882352941176
F1-Score:  0.7810650887573964

📋 Confusion Matrix:


array([[1897,   35],
       [   2,   66]])

## Load Saved Model
Define a function to load the previously saved model checkpoint and recreate the trained model with its metadata.

In [69]:
# LOAD THE SAVED MODEL

def load_model(model_path='/content/drive/MyDrive/Colab Notebooks/Predictive Maintenance Dataset (AI4I 2020)/best_predictive_model.pth'):
    """Load the saved model and return model object with metadata."""

    # Load checkpoint
    checkpoint = torch.load(model_path, map_location=device)

    # Recreate model architecture
    model = PredictiveHyperANN(
        input_dim=checkpoint['input_dim'],
        output_dim=1,
        num_hidden_layes=int(checkpoint['model_params']['num_hidden_layers']),
        neurons_per_layers=int(checkpoint['model_params']['neurons_per_layer']),
        dropout_rate=checkpoint['model_params']['dropout_rate']
    ).to(device)

    # Load trained weights
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()

    print(f"✅ Model loaded from '{model_path}'")
    print(f"   - Accuracy: {checkpoint['accuracy']:.4f}")
    print(f"   - Hidden Layers: {checkpoint['model_params']['num_hidden_layers']}")
    print(f"   - Neurons per Layer: {checkpoint['model_params']['neurons_per_layer']}")
    print(f"   - Dropout Rate: {checkpoint['model_params']['dropout_rate']}")

    return model, checkpoint

# Load the model
loaded_model, model_metadata = load_model()

✅ Model loaded from '/content/drive/MyDrive/Colab Notebooks/Predictive Maintenance Dataset (AI4I 2020)/best_predictive_model.pth'
   - Accuracy: 0.9985
   - Hidden Layers: 6
   - Neurons per Layer: 128
   - Dropout Rate: 0.1


## Prediction Pipeline
Create a function to predict machine failure for new input data. This handles data preprocessing, feature encoding, scaling, and model inference.

In [71]:
# PREDICTION PIPELINE

def predict_machine_failure(input_data, model, checkpoint, device='cpu'):
    """
    Predict machine failure for new input data.

    Parameters:
    -----------
    input_data : dict or DataFrame
        Input features (Type, Air temperature, Process temperature,
        Rotational speed, Torque, Tool wear, TWF, HDF, PWF, OSF, RNF)
    model : nn.Module
        Trained PyTorch model
    checkpoint : dict
        Model metadata (scaler params, feature columns, etc.)
    device : str
        Device to run inference on

    Returns:
    --------
    dict : Prediction result with probability and class
    """

    # Convert to DataFrame if dict
    if isinstance(input_data, dict):
        input_df = pd.DataFrame([input_data])
    else:
        input_df = input_data.copy()

    # Encode 'Type' column
    type_encoder = LabelEncoder()
    type_encoder.classes_ = np.array(checkpoint['type_encoder_classes'])

    if 'Type' in input_df.columns:
        try:
            input_df['Type'] = type_encoder.transform(input_df['Type'])
        except ValueError as e:
            raise ValueError(f"Invalid 'Type' value. Must be one of {type_encoder.classes_}. Error: {e}")

    # Ensure correct column order
    feature_columns = checkpoint['feature_columns']
    input_df = input_df[feature_columns]

    # Scale features
    scaler_mean = np.array(checkpoint['scaler_mean'])
    scaler_scale = np.array(checkpoint['scaler_scale'])
    input_scaled = (input_df.values - scaler_mean) / scaler_scale

    # Convert to tensor
    input_tensor = torch.Tensor(input_scaled,dtype=torch.float32).to(device)

    # Predict
    model.eval()
    with torch.no_grad():
        logits = model(input_tensor).view(-1)
        probabilities = torch.sigmoid(logits).cpu().numpy()
        predictions = (probabilities >= 0.5).astype(int)

    # Format results
    results = []
    for i, (prob, pred) in enumerate(zip(probabilities, predictions)):
        results.append({
            'Sample': i + 1,
            'Failure_Probability': float(prob),
            'Prediction': 'Failure' if pred == 1 else 'No Failure',
            'Confidence': f"{abs(prob - 0.5) * 2 * 100:.2f}%"
        })

    return pd.DataFrame(results)

## Example: Single Prediction
Demonstrate how to make a single prediction using a dictionary input with sample feature values.

In [72]:
# EXAMPLE: PREDICT WITH USER INPUT

# Example 1: Single prediction with dictionary
sample_input = {
    'Type': 'L',                          # Product type: L, M, or H
    'Air temperature [K]': 298.5,        # Air temperature in Kelvin
    'Process temperature [K]': 308.6,    # Process temperature in Kelvin
    'Rotational speed [rpm]': 1500,      # Rotational speed
    'Torque [Nm]': 45.0,                 # Torque in Nm
    'Tool wear [min]': 10,               # Tool wear in minutes
    'TWF': 0,                            # Tool Wear Failure (0 or 1)
    'HDF': 0,                            # Heat Dissipation Failure (0 or 1)
    'PWF': 0,                            # Power Failure (0 or 1)
    'OSF': 0,                            # Overstrain Failure (0 or 1)
    'RNF': 0                             # Random Failure (0 or 1)
}

print("=" * 60)
print("SINGLE PREDICTION EXAMPLE")
print("=" * 60)
result = predict_machine_failure(sample_input, loaded_model, model_metadata, device)
display(result)

SINGLE PREDICTION EXAMPLE


,Sample,Failure_Probability,Prediction,Confidence
0,1,0.153801,No Failure,69.24%


## Example: Batch Predictions
Demonstrate how to make multiple predictions at once using a DataFrame input.

In [73]:
# EXAMPLE: MULTIPLE PREDICTIONS

# Example 2: Batch prediction with DataFrame
batch_input = pd.DataFrame({
    'Type': ['L', 'M', 'H'],
    'Air temperature [K]': [298.2, 300.5, 299.8],
    'Process temperature [K]': [308.7, 310.2, 309.5],
    'Rotational speed [rpm]': [1450, 1600, 1550],
    'Torque [Nm]': [42.0, 50.5, 47.3],
    'Tool wear [min]': [5, 100, 50],
    'TWF': [0, 1, 0],
    'HDF': [0, 0, 1],
    'PWF': [0, 0, 0],
    'OSF': [0, 1, 0],
    'RNF': [0, 0, 0]
})

print("=" * 60)
print("BATCH PREDICTION EXAMPLE")
print("=" * 60)
print("\nInput Data:")
display(batch_input)

print("\nPredictions:")
batch_result = predict_machine_failure(batch_input, loaded_model, model_metadata, device)
display(batch_result)

BATCH PREDICTION EXAMPLE

Input Data:


,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],TWF,HDF,PWF,OSF,RNF
0,L,298.2,308.7,1450,42.0,5,0,0,0,0,0
1,M,300.5,310.2,1600,50.5,100,1,0,0,1,0
2,H,299.8,309.5,1550,47.3,50,0,1,0,0,0



Predictions:


,Sample,Failure_Probability,Prediction,Confidence
0,1,0.148285,No Failure,70.34%
1,2,1.000000,Failure,100.00%
2,3,0.999166,Failure,99.83%


## Interactive Prediction Interface
An optional interactive function that allows users to input machine parameters manually and get real-time predictions.

In [75]:
# INTERACTIVE PREDICTION (Optional)
def interactive_prediction(model, checkpoint, device):
    """Interactive prediction using input() for user-friendly predictions."""

    print("\n" + "=" * 60)
    print("INTERACTIVE MACHINE FAILURE PREDICTION")
    print("=" * 60)
    print("\nEnter the following values:")

    try:
        user_input = {
            'Type': input("Type (L/M/H): ").strip().upper(),
            'Air temperature [K]': float(input("Air temperature [K]: ").strip()),
            'Process temperature [K]': float(input("Process temperature [K]: ").strip()),
            'Rotational speed [rpm]': int(input("Rotational speed [rpm]: ").strip()),
            'Torque [Nm]': float(input("Torque [Nm]: ").strip()),
            'Tool wear [min]': int(input("Tool wear [min]: ").strip()),
            'TWF': int(input("TWF (0/1): ").strip()),
            'HDF': int(input("HDF (0/1): ").strip()),
            'PWF': int(input("PWF (0/1): ").strip()),
            'OSF': int(input("OSF (0/1): ").strip()),
            'RNF': int(input("RNF (0/1): ").strip())
        }

        result = predict_machine_failure(user_input, model, checkpoint, device)

        print("\n" + "=" * 60)
        print("PREDICTION RESULT")
        print("=" * 60)
        display(result)

    except ValueError as e:
        print(f"\n❌ Invalid input: {e}")
    except Exception as e:
        print(f"\n❌ Error: {e}")

# Uncomment to run interactive prediction
interactive_prediction(loaded_model, model_metadata, device)


INTERACTIVE MACHINE FAILURE PREDICTION

Enter the following values:
Type (L/M/H): L
Air temperature [K]: 298.2
Process temperature [K]: 308.7
Rotational speed [rpm]: 1450
Torque [Nm]: 42.0
Tool wear [min]: 5
TWF (0/1): 0
HDF (0/1): 0
PWF (0/1): 0
OSF (0/1): 0
RNF (0/1): 0

PREDICTION RESULT


,Sample,Failure_Probability,Prediction,Confidence
0,1,0.148285,No Failure,70.34%
